<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 15



<h2 style="color:DodgerBlue">Описание проекта:</h2>

Описание задачи:
Создать базовый класс OrderLine в C#, который будет представлять информацию о
строке заказа, содержащей детали одного товара в заказе. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.
Требования к базовому классу OrderLine:
• Атрибуты: ID товара (ProductId), Название товара (ProductName), Цена
товара (Price).
• Методы:
o
o CalculateTotal(): метод для расчета общей стоимости строки заказа.
o UpdatePrice(decimal newPrice): метод для обновления цены товара в
строке заказа.
o GetProductDetails(): метод для получения деталей товара.
Требования к производным классам:
1. СтандартнаяСтрока (StandardLine): Должна содержать дополнительные
атрибуты, такие как Количество единиц (Units). Метод CalculateTotal() должен
быть переопределен для учета количества единиц при расчете общей
стоимости.
2. СпециальнаяСтрока (SpecialLine): Должна содержать дополнительные
атрибуты, такие как Скидка (Discount). Метод UpdatePrice() должен быть
переопределен для применения скидки к цене товара.
3. БесплатнаяСтрока (FreeLine) (если требуется третий класс): Должна
содержать дополнительные атрибуты, такие как Предварительный платеж
(Prepayment). Метод CalculateTotal() должен быть переопределен для учета
предварительного плата при расчете общей стоимости.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:

using System;
using System.Collections.Generic;
using System.Linq;

public interface IManufacturerInfo
{
    string GetManufacturerInfo();
}

public interface IContactInfo
{
    string GetContactInfo();
}

public interface IStorageInfo
{
    string GetStorageInfo();
}

// Делегаты
public delegate void DisplayDelegate();
public delegate void PriceChangeDelegate(decimal oldPrice, decimal newPrice);
public delegate void ProductUpdateDelegate(string updateMessage);

public class OrderLine : IManufacturerInfo, IContactInfo
{
    // Существующие свойства
    public int ProductId { get; private set; }
    public string ProductName { get; private set; }
    public decimal Price { get; protected set; }
    public string Manufacturer { get; private set; }
    public string Country { get; private set; }
    public string Category { get; set; }
    public List<string> Tags { get; private set; }
    public int QuantitySold { get; set; }
    public string ContactEmail {get;set;}
    public string ContactPhone {get;set;}
    public DateTime CreatedDate { get; private set; }
    public event Action<string> PriceUpdated;
    public event PriceChangeDelegate OnPriceChange;
    public event ProductUpdateDelegate OnProductUpdate;

    protected virtual void OnProductUpdateInvoke(string message)
    {
        OnProductUpdate?.Invoke(message);
    }

    public OrderLine(int productId, string productName, decimal price, string manufacturer, string country, string contactEmail, string contactPhone, string category = "General")
    {
        
        if (productId <= 0)
            throw new ArgumentException("ProductId должен быть положительным числом.", nameof(productId));
        ProductId = productId;
        ProductName = productName;
        if (price < 0)
            throw new ArgumentException("Price не может быть отрицательным.", nameof(price));
        Price = price;
        Manufacturer = manufacturer;
        Country = country;
        ContactEmail = contactEmail;
        ContactPhone = contactPhone;
        Category = category;
        CreatedDate = DateTime.Now;
        Tags = new List<string>();
        QuantitySold = 0;
    }

    public virtual void UpdatePrice(decimal newPrice)
    {
        if (newPrice + Price < 0)
            throw new ArgumentException("Price не может быть отрицательным.", nameof(newPrice));
        
        decimal oldPrice = Price;
        Price += newPrice;
        PriceUpdated?.Invoke($"Цена обновлена: {Price}");
        OnPriceChange?.Invoke(oldPrice, Price);
    }

    public virtual decimal CalculateTotal()
    {
        return Price;
    }

    public virtual string GetProductDetails()
    {
        return $"ID: {ProductId}, Название: {ProductName}, Цена: {Price}, Категория: {Category}, Дата создания: {CreatedDate:dd.MM.yyyy}";
    }

    public string GetManufacturerInfo()
    {
        return $"Производитель: {Manufacturer}, Страна: {Country}";
    }

    public string GetContactInfo()
    {
        return $"Email: {ContactEmail}, Телефон: {ContactPhone}";
    }

    // Новые методы
    public void AddTag(string tag)
    {
        if (!Tags.Contains(tag))
        {
            Tags.Add(tag);
            OnProductUpdateInvoke($"Добавлен тег: {tag}");
        }
    }

    public void RemoveTag(string tag)
    {
        if (Tags.Contains(tag))
        {
            Tags.Remove(tag);
            OnProductUpdateInvoke($"Удален тег: {tag}");
        }
    }

    public List<string> GetTags()
    {
        return new List<string>(Tags);
    }

    public void IncrementQuantitySold(int amount = 1)
    {
        if (amount > 0)
        {
            QuantitySold += amount;
            OnProductUpdateInvoke($"Обновлено количество продаж: {QuantitySold}");
        }
    }
}

public class StandardLine : OrderLine, IStorageInfo
{
    // Существующие свойства
    public int Units { get; private set; }
    public string StorageLocation { get; private set; }
    public string UnitType { get; private set; }
    
    // Новые свойства
    public DateTime LastRestockDate { get; set; }
    public int MinStockLevel { get; set; }
    public int MaxStockLevel { get; set; }
    public List<DateTime> RestockHistory { get; private set; }

    

    public StandardLine(int productId, string productName, decimal price, string manufacturer, string country, 
    string contactEmail, string contactPhone, int units, string storageLocation, string unitType, 
    string category = "General", int minStock = 10, int maxStock = 100)
        : base(productId, productName, price, manufacturer, country, contactEmail, contactPhone, category)
    {
        if (units <= 0)
            throw new ArgumentException("Units должен быть положительным числом.", nameof(units));
        Units = units;
        StorageLocation = storageLocation;
        UnitType = unitType;
        MinStockLevel = minStock;
        MaxStockLevel = maxStock;
        RestockHistory = new List<DateTime>();
        LastRestockDate = DateTime.Now;
        RestockHistory.Add(LastRestockDate);
    }

    public override decimal CalculateTotal()
    {
        return Price * Units;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() + $", Количество: {Units} {UnitType}, Место хранения: {StorageLocation}";
    }

    public string GetStorageInfo()
    {
        return $"Место хранения: {StorageLocation}, Уровень запаса: {Units}, Минимальный уровень: {MinStockLevel}, Максимальный уровень: {MaxStockLevel}";
    }

    // Новые методы
    public void Restock(int newUnits)
    {
        if (newUnits > 0)
        {
            Units += newUnits;
            LastRestockDate = DateTime.Now;
            RestockHistory.Add(LastRestockDate);
            OnProductUpdateInvoke($"Пополнение запаса: +{newUnits} единиц");
        }
    }

    public bool IsLowStock()
    {
        return Units <= MinStockLevel;
    }

    public bool IsOverStock()
    {
        return Units >= MaxStockLevel;
    }

    public List<DateTime> GetRestockHistory()
    {
        return new List<DateTime>(RestockHistory);
    }
}

public class SpecialLine : StandardLine
{
    // Существующие свойства
    public decimal Discount { get; private set; }
    public string DiscountReason { get; private set; }
    public string DiscountCode { get; private set; }
    
    // Новые свойства
    public DateTime ValidFrom { get; private set; }
    public DateTime ValidTo { get; private set; }
    public List<string> EligibleProducts { get; private set; }
    public int UsageCount { get; private set; }



    public SpecialLine(int productId, string productName, decimal price, string manufacturer, string country, string contactEmail,
     string contactPhone, int units, string storageLocation, string unitType, decimal discount, string discountReason, 
     string discountCode, DateTime validFrom, DateTime validTo, string category = "General", int minStock = 10, int maxStock = 100)
        : base( productId, productName, price,  manufacturer,  country, 
        contactEmail, contactPhone,  units,  storageLocation, unitType, 
        category ,  minStock ,  maxStock )
    {
        if (discount < 0)
            throw new ArgumentException("Discount не может быть отрицательным.", nameof(discount));
        Discount = discount;
        DiscountReason = discountReason;
        DiscountCode = discountCode;
        ValidFrom = validFrom;
        ValidTo = validTo;
        EligibleProducts = new List<string>();
        UsageCount = 0;
    }

    public override decimal CalculateTotal()
    {
        return (Price * Units) - Discount;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() + $", Скидка: {Discount}, Причина: {DiscountReason}, Код: {DiscountCode}, Действует с {ValidFrom:dd.MM.yyyy} по {ValidTo:dd.MM.yyyy}";
    }

    public string GetDiscountInfo()
    {
        return $"Скидка: {Discount}, Причина: {DiscountReason}, Код: {DiscountCode}, Действует с {ValidFrom:dd.MM.yyyy} по {ValidTo:dd.MM.yyyy}";
    }

    // Новые методы
    public bool IsDiscountValid()
    {
        DateTime now = DateTime.Now;
        return now >= ValidFrom && now <= ValidTo;
    }

    public void ApplyDiscountToProduct(string productId)
    {
        if (!EligibleProducts.Contains(productId))
        {
            EligibleProducts.Add(productId);
            OnProductUpdateInvoke($"Скидка применена к продукту: {productId}");
        }
    }

    public List<string> GetEligibleProducts()
    {
        return new List<string>(EligibleProducts);
    }

    public void IncrementUsage()
    {
        UsageCount++;
        OnProductUpdateInvoke($"Скидка использована, всего использований: {UsageCount}");
    }
}

public class FreeLine : SpecialLine
{
    protected decimal Prepayment { get; set; }
    public string PrepaymentType { get; private set; }
    public string GiftMessage { get; private set; }
    
    // Новые свойства
    public List<string> RecipientEmails { get; private set; }
    public string CertificateCode { get; private set; }
    public DateTime CertificateExpiration { get; private set; }
    public bool IsRedeemed { get; private set; }

    public FreeLine(int productId, string productName, decimal price, string manufacturer, string country, string contactEmail, string contactPhone, int units, string storageLocation, string unitType, decimal discount, string discountReason, string discountCode, decimal prepayment, string prepaymentType, string giftMessage, DateTime validFrom, DateTime validTo, string certificateCode, DateTime certExpiration, string category = "General", int minStock = 10, int maxStock = 100)
        : base(productId, productName, price, manufacturer, country, contactEmail, contactPhone, units, storageLocation, unitType, discount, discountReason, discountCode, validFrom, validTo, category, minStock, maxStock)
    {
        if (prepayment < 0)
            throw new ArgumentException("Prepayment не может быть отрицательным.", nameof(prepayment));
        Prepayment = prepayment;
        PrepaymentType = prepaymentType;
        GiftMessage = giftMessage;
        CertificateCode = certificateCode;
        CertificateExpiration = certExpiration;
        RecipientEmails = new List<string>();
        IsRedeemed = false;
    }

    public override decimal CalculateTotal()
    {
        return Price * Units - Discount - Prepayment;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() + $", Предоплата: {Prepayment} ({PrepaymentType}), Сообщение: {GiftMessage}, Сертификат: {CertificateCode}";
    }

    public string GetPrepaymentInfo()
    {
        return $"Предоплата: {Prepayment}, Тип: {PrepaymentType}";
    }

    public string GetGiftMessage()
    {
        return $"Подарочное сообщение: {GiftMessage}";
    }

    // Новые методы
    public void AddRecipientEmail(string email)
    {
        if (!RecipientEmails.Contains(email))
        {
            RecipientEmails.Add(email);
            OnProductUpdateInvoke($"Добавлен получатель: {email}");
        }
    }

    public List<string> GetRecipients()
    {
        return new List<string>(RecipientEmails);
    }

    public void RedeemCertificate()
    {
        if (!IsRedeemed && DateTime.Now <= CertificateExpiration)
        {
            IsRedeemed = true;
            OnProductUpdateInvoke($"Сертификат {CertificateCode} активирован");
        }
        else if (DateTime.Now > CertificateExpiration)
        {
            OnProductUpdateInvoke($"Сертификат {CertificateCode} просрочен");
        }
    }

    public bool IsValidCertificate()
    {
        return !IsRedeemed && DateTime.Now <= CertificateExpiration;
    }
}

public class OrderCollection<T> where T : OrderLine
{
    private List<T> _orderLines = new List<T>();
    private Dictionary<string, List<T>> _categoryGroups = new Dictionary<string, List<T>>();

    public void AddOrderLine(T orderLine)
    {
        _orderLines.Add(orderLine);
        
        // Группировка по категориям
        if (!_categoryGroups.ContainsKey(orderLine.Category))
        {
            _categoryGroups[orderLine.Category] = new List<T>();
        }
        _categoryGroups[orderLine.Category].Add(orderLine);
    }

    public void RemoveOrderLine(T orderLine)
    {
        _orderLines.Remove(orderLine);
        
        // Удаление из группировки
        if (_categoryGroups.ContainsKey(orderLine.Category))
        {
            _categoryGroups[orderLine.Category].Remove(orderLine);
            if (_categoryGroups[orderLine.Category].Count == 0)
            {
                _categoryGroups.Remove(orderLine.Category);
            }
        }
    }

    public decimal CalculateTotal()
    {
        decimal total = 0;
        foreach (var line in _orderLines)
        {
            total += line.CalculateTotal();
        }
        return total;
    }

    public void PrintOrderDetails()
    {
        foreach (var line in _orderLines)
        {
            Console.WriteLine(line.GetProductDetails());
        }
    }

    // Новые методы
    public List<T> GetByCategory(string category)
    {
        if (_categoryGroups.ContainsKey(category))
        {
            return new List<T>(_categoryGroups[category]);
        }
        return new List<T>();
    }

    public Dictionary<string, List<T>> GetGroupedByCategory()
    {
        return new Dictionary<string, List<T>>(_categoryGroups);
    }

    public List<T> GetByTag(string tag)
    {
        return _orderLines.Where(line => line.GetTags().Contains(tag)).ToList();
    }

    public List<T> GetSortedByPrice(bool ascending = true)
    {
        return ascending ? 
            _orderLines.OrderBy(line => line.Price).ToList() : 
            _orderLines.OrderByDescending(line => line.Price).ToList();
    }

    public decimal GetTotalByCategory(string category)
    {
        if (_categoryGroups.ContainsKey(category))
        {
            return _categoryGroups[category].Sum(line => line.CalculateTotal());
        }
        return 0;
    }
}

public class WriteCalculateCol
{
    private readonly FreeLine freeLine;

    public WriteCalculateCol(FreeLine freeLine)
    {
        this.freeLine = freeLine;
    }
    
    public void WriteDetails()
    {
        Console.WriteLine(freeLine.GetProductDetails());
        Console.Write(freeLine.GetManufacturerInfo());
        Console.WriteLine(freeLine.GetContactInfo());
        Console.Write(freeLine.GetStorageInfo());
        Console.WriteLine(freeLine.GetDiscountInfo());
        Console.Write(freeLine.GetPrepaymentInfo());
        Console.WriteLine(freeLine.GetGiftMessage());
    }
}

// Пример использования
OrderLine orderLine = new OrderLine(1, "Продукт", 100.0m, "ООО Рога и Копыта", "Россия", "info@roga.ru", "+7-900-000-00-00", "Электроника");
orderLine.AddTag("популярный");
orderLine.IncrementQuantitySold(5);
Console.WriteLine(orderLine.GetProductDetails());
Console.WriteLine(orderLine.GetManufacturerInfo());
Console.WriteLine(orderLine.GetContactInfo());

StandardLine standardLine = new StandardLine(2, "Стандартный продукт", 50.0m, "ООО Бумага+", "Беларусь", "contact@bumaga.by", "+375-29-111-22-33", 3, "Склад 1", "шт", "Канцелярия");
standardLine.Restock(10);
Console.WriteLine(standardLine.GetProductDetails());
Console.WriteLine(standardLine.GetManufacturerInfo());
Console.WriteLine(standardLine.GetContactInfo());
Console.WriteLine(standardLine.GetStorageInfo());

SpecialLine specialLine = new SpecialLine(3, "Специальный продукт", 80.0m, "ООО Пиши+", "Китай", "service@pishi.cn", "+86-10-1234-5678", 2, "Склад 2", "шт", 15.0m, "Акция", "DISCOUNT15", DateTime.Now, DateTime.Now.AddDays(30), "Техника");
specialLine.ApplyDiscountToProduct("PRD001");
Console.WriteLine(specialLine.GetProductDetails());
Console.WriteLine(specialLine.GetManufacturerInfo());
Console.WriteLine(specialLine.GetContactInfo());
Console.WriteLine(specialLine.GetStorageInfo());
Console.WriteLine(specialLine.GetDiscountInfo());

FreeLine freeLine = new FreeLine(4, "Бесплатный продукт", 200.0m, "ООО Подаркофф", "Германия", "gift@podarkoff.de", "+49-30-9876-5432", 1, "Склад 3", "шт", 50.0m, "Праздник", "GIFT50", 30.0m, "Бонус", "С праздником!", DateTime.Now, DateTime.Now.AddDays(90), "CERT001", DateTime.Now.AddYears(1), "Подарки");
freeLine.AddRecipientEmail("recipient@example.com");
freeLine.RedeemCertificate();
Console.WriteLine(freeLine.GetProductDetails());
Console.WriteLine(freeLine.GetManufacturerInfo());
Console.WriteLine(freeLine.GetContactInfo());
Console.WriteLine(freeLine.GetStorageInfo());
Console.WriteLine(freeLine.GetDiscountInfo());
Console.WriteLine(freeLine.GetPrepaymentInfo());
Console.WriteLine(freeLine.GetGiftMessage());

OrderCollection<OrderLine> orderCollection = new OrderCollection<OrderLine>();
orderCollection.AddOrderLine(orderLine);
orderCollection.PrintOrderDetails();

OrderCollection<StandardLine> standardCollection = new OrderCollection<StandardLine>();
standardCollection.AddOrderLine(standardLine);
standardCollection.PrintOrderDetails();

OrderCollection<SpecialLine> specialCollection = new OrderCollection<SpecialLine>();
specialCollection.AddOrderLine(specialLine);
specialCollection.PrintOrderDetails();

OrderCollection<FreeLine> freeCollection = new OrderCollection<FreeLine>();
freeCollection.AddOrderLine(freeLine);
freeCollection.PrintOrderDetails();

// Примеры использования новых методов коллекции
var electronics = orderCollection.GetByCategory("Электроника");
Console.WriteLine($"Продукты в категории 'Электроника': {electronics.Count}");

var sortedByPrice = orderCollection.GetSortedByPrice(false);
Console.WriteLine("Сортировка по цене (по убыванию):");
foreach (var item in sortedByPrice)
{
    Console.WriteLine(item.GetProductDetails());
}

WriteCalculateCol writeCalculateCol = new WriteCalculateCol(freeLine);
writeCalculateCol.WriteDetails();

OrderLine orderLine1 = new OrderLine(5, "Новый продукт", 120.0m, "ООО Новый", "США", "new@new.com", "+1-800-555-1234");
DisplayDelegate displayDelegate = () => Console.WriteLine(orderLine1.GetProductDetails());
orderLine1.PriceUpdated += message => Console.WriteLine(message);
orderLine1.OnPriceChange += (oldPrice, newPrice) => Console.WriteLine($"Цена изменилась с {oldPrice} на {newPrice}");
orderLine1.OnProductUpdate += msg => Console.WriteLine($"Обновление: {msg}");

ID: 1, Название: Продукт, Цена: 100,0, Категория: Электроника, Дата создания: 17.11.2025
Производитель: ООО Рога и Копыта, Страна: Россия
Производитель: ООО Рога и Копыта, Страна: Россия
Email: info@roga.ru, Телефон: +7-900-000-00-00
Email: info@roga.ru, Телефон: +7-900-000-00-00
ID: 2, Название: Стандартный продукт, Цена: 50,0, Категория: Канцелярия, Дата создания: 17.11.2025, Количество: 13 шт, Место хранения: Склад 1
ID: 2, Название: Стандартный продукт, Цена: 50,0, Категория: Канцелярия, Дата создания: 17.11.2025, Количество: 13 шт, Место хранения: Склад 1
Производитель: ООО Бумага+, Страна: Беларусь
Производитель: ООО Бумага+, Страна: Беларусь
Email: contact@bumaga.by, Телефон: +375-29-111-22-33
Email: contact@bumaga.by, Телефон: +375-29-111-22-33
Место хранения: Склад 1, Уровень запаса: 13, Минимальный уровень: 10, Максимальный уровень: 100
Место хранения: Склад 1, Уровень запаса: 13, Минимальный уровень: 10, Максимальный уровень: 100
ID: 3, Название: Специальный продукт, Цена: 8